# SpeechRecognition

## 0.Introducción al Reconocimiento de Voz con Python

El reconocimiento de voz es una tecnología que permite convertir el habla en texto de forma automática. En este notebook, exploraremos cómo podemos implementar esta funcionalidad utilizando Python y algunas de sus bibliotecas especializadas. 

### ¿Qué es el reconocimiento de voz?
El reconocimiento de voz es el proceso mediante el cual un sistema informático interpreta y transcribe el lenguaje hablado. Esto tiene aplicaciones en asistentes virtuales, transcripción de audio, sistemas de control por voz y más.

### Objetivos del Notebook
- **Comprender el funcionamiento básico:** Conocer cómo se procesa el audio y se convierte en texto.
- **Aprender a utilizar bibliotecas:** Trabajar con librerías como SpeechRecognition y PyAudio para capturar y procesar el audio.
- **Aplicaciones prácticas:** Ver ejemplos reales de cómo implementar y ajustar un sistema de reconocimiento de voz en Python.

A lo largo del notebook, abordaremos desde la instalación y configuración del entorno, pasando por ejemplos básicos de reconocimiento, hasta ejercicios prácticos que os permitirán experimentar y profundizar en el tema.


## 1. Introducción Teórica

El reconocimiento de voz es una tecnología fascinante que ha evolucionado enormemente en las últimas décadas. Antes de adentrarnos en la práctica, es útil conocer algunos conceptos básicos y los hitos que han marcado su desarrollo:

### Conceptos Básicos
- **Procesamiento de Señales:** Se refiere a cómo se captura y transforma el audio en datos que el ordenador puede analizar.
- **Modelos Acústicos y de Lenguaje:** Son fundamentales para que el sistema entienda y transcriba el habla, permitiendo convertir sonidos en palabras.

### Hitos Importantes
- **Primeros Sistemas de Reconocimiento:** En los años 50 y 60 se realizaron los primeros experimentos en reconocimiento de voz. Estos sistemas pioneros sentaron las bases para lo que vendría. [Más información](https://en.wikipedia.org/wiki/Speech_recognition).
- **Introducción de Modelos Ocultos de Markov (HMM):** Durante las décadas siguientes, los HMM revolucionaron el campo, mejorando la precisión del reconocimiento. [Descubre cómo](https://en.wikipedia.org/wiki/Hidden_Markov_model).
- **Avances con Deep Learning:** En tiempos recientes, el uso de redes neuronales profundas ha permitido alcanzar niveles de precisión y eficiencia sin precedentes. [Lee sobre el impacto del deep learning](https://en.wikipedia.org/wiki/Deep_learning).

Con estos conceptos, se sienta el escenario para entender cómo el reconocimiento de voz ha pasado de ser una idea experimental a una herramienta clave en diversas aplicaciones modernas.

## 2.Instalación y Configuración del Entorno


Antes de arrancar, aseguraos de tener instaladas estas dependencias:

- **[SpeechRecognition](https://pypi.org/project/SpeechRecognition/):** La biblioteca principal para reconocimiento de voz en Python. Es una inferfaz que permite usar los siguientes motores/APIs:
    - CMU Sphinx (works offline)
    - Google Speech Recognition
    - Google Cloud Speech API
    - Wit.ai
    - Microsoft Azure Speech
    - Microsoft Bing Voice Recognition (Deprecated)
    - Houndify API
    - IBM Speech to Text
    - Snowboy Hotword Detection (works offline)
    - Tensorflow
    - Vosk API (works offline)
    - OpenAI whisper (works offline)
    - OpenAI Whisper API
    - Groq Whisper API
    - Cohere Transcribe API
- **[PyAudio](https://pypi.org/project/PyAudio/):** Para acceder al micrófono y trabajar con audio en tiempo real (requiere Python 3.12).
    - *Alternativa:* Si tenéis líos con PyAudio, podéis usar [SoundDevice](https://pypi.org/project/sounddevice/) junto a [Wavio](https://pypi.org/project/wavio/) para capturar audio.
- (opcional) **[simpleaudio](https://pypi.org/project/simpleaudio/):** Para reproducir audio.
- (opcional) **[IPython](https://pypi.org/project/ipython/):** Para reproducir audio (y mucho más) en este notebook.

In [ ]:
!pip install SpeechRecognition pyaudio IPython

Para comprobar que funcionan y para el resto del notebook podemos importarlas

In [3]:
import speech_recognition as sr # Usaremos la pyaudio para abrir el microfono
from IPython.display import Audio, display # Para gestionar y reproducir audio en Jupyter (sugerencia de Pedro Antonio Prieto)
import os # Para el acceso al sistema de ficheros

## 3.Primeros Pasos: Reconocimiento Básico de Voz


### 3.1 Abrir el microfono

En esta sección vamos a abrir el micrófono, grabar lo que se diga y luego reproducir la grabación. El siguiente código muestra cómo hacerlo:


Lo primero es buscar un micrófono de nuestro ordenador para poder reconocer la voz 

In [4]:
# Creamos un reconocedor de voz
r = sr.Recognizer()

El reconocedor puede transcribir audio desde un fichero previamente guardado o capturar el audio directamente del micrófono. Para la captura desde micrófono hay que instanciarlo dentro de un contexto **[with](https://www.geeksforgeeks.org/python/with-statement-in-python/)**.

```python
with sr.Microphone() as source:
```

Dentro del bloque *with* realizamos las tareas necesarias con el micrófono. El primer paso siempre ha de ser la detección del nivel de ruido ambiental que le permita distinguir los momentos de silencio mediante **[adjust_for_ambient_noise(source, duration=1)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instanceadjust_for_ambient_noisesource-audiosource-duration-float--1---none)** (por defecto escucha durante un segundo para detectar el nivel del ruido).

```python
    r.adjust_for_ambient_noise(source)
```

Tras esto la captura de audio la hacemos mediante **[listen(source)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancelistensource-audiosource-timeout-unionfloat-none--none-phrase_time_limit-unionfloat-none--none-snowboy_configuration-uniontuplestr-iterablestr-none--none-stream-bool--false---audiodata)** que captura audio hasta que encuentra un silencio y lo devuelve como un objeto de tipo [AudioData](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#audiodataframe_data-bytes-sample_rate-int-sample_width-int---audiodata).

In [5]:
# Iniciar la captura del micrófono
with sr.Microphone() as source:
    # Ajustar el recognizer al ruido ambiental
    r.adjust_for_ambient_noise(source)

    print("Capturando audio. ¡Hable ahora!")
    audio = r.listen(source)

Capturando audio. ¡Hable ahora!


### 3.2 Guardar y reproducir audio (opcional)


Primero escogemos la carpeta en la que guardaremos el audio.

In [6]:
# Nombre de la carpeta
folder = "media/audio"

# Verifica si la carpeta no existe; si no existe, la crea
if not os.path.exists(folder):
    os.makedirs(folder)

Luego podemos guardar audio en los siguientes formatos RAW, WAW, AIFF, FLAC

In [8]:
# Guardar audio en un archivo RAW
raw_path = os.path.join(folder, "microphone-results.raw")
with open(raw_path, "wb") as f:
    f.write(audio.get_raw_data())
print(f"Grabación guardada en '{raw_path}'.")

# Guardar audio en un archivo WAV
wav_path = os.path.join(folder, "microphone-results.wav")
with open(wav_path, "wb") as f:
    f.write(audio.get_wav_data())
print(f"Grabación guardada en '{wav_path}'.")

# Guardar audio en un archivo AIFF
aiff_path = os.path.join(folder, "microphone-results.aiff")
with open(aiff_path, "wb") as f:
    f.write(audio.get_aiff_data())
print(f"Grabación guardada en '{aiff_path}'.")

# Guardar audio en un archivo FLAC
flac_path = os.path.join(folder, "microphone-results.flac")
with open(flac_path, "wb") as f:
    f.write(audio.get_flac_data())
print(f"Grabación guardada en '{flac_path}'.")


Grabación guardada en 'media/audio\microphone-results.raw'.
Grabación guardada en 'media/audio\microphone-results.wav'.
Grabación guardada en 'media/audio\microphone-results.aiff'.
Grabación guardada en 'media/audio\microphone-results.flac'.


Una vez guardado el archivo podemos reproducirlo de la siguiente forma

In [9]:
display(Audio("media/audio/microphone-results.wav", autoplay=False))

Una vez guardado el audio se puede cargar de nuevo en memoria por si lo necesitasemos

In [17]:
from pathlib import Path

# Usamos el directorio actual de trabajo
base_dir = Path.cwd()
audio_path = base_dir / "media" / "audio" / "microphone-results.wav"

# Verificamos que el archivo existe
if not audio_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo de audio en: {audio_path}")

# Abrimos el archivo de audio y lo convertimos a datos que puedan procesarse
with sr.AudioFile(str(audio_path)) as source:
    audio = r.record(source)  # Leemos todo el archivo

## 4. Reconocimiento de audio

Una vez que hemos capturado un audio podemos proceder al reconocimiento, es decir, la obtención del texto a partir del audio. Para ello disponemos de diversos métodos en función del motor o API que deseemos emplear:
- [recognize_sphinx(audio_data)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancerecognize_sphinxaudio_data-audiodata-language-str--en-us-keyword_entries-unioniterabletuplestr-float-none--none-grammar-unionstr-none--none-show_all-bool--false---unionstr-pocketsphinxpocketsphinxdecoder)
- [recognize_google(audio_data)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancerecognize_googleaudio_data-audiodata-key-unionstr-none--none-language-str--en-us--pfilter-union0-1-show_all-bool--false---unionstr-dictstr-any)
- [recognize_google_cloud(audio_data)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancerecognize_google_cloudaudio_data-audiodata-credentials_json_path-unionstr-none--none-kwargs---unionstr-dictstr-any)
- [recognize_wit(audio_data)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancerecognize_witaudio_data-audiodata-key-str-show_all-bool--false---unionstr-dictstr-any)
- [recognize_bing(audio_data)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancerecognize_bingaudio_data-audiodata-key-str-language-str--en-us-show_all-bool--false---unionstr-dictstr-any)
- [recognize_houndify(audio_data)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancerecognize_houndifyaudio_data-audiodata-client_id-str-client_key-str-show_all-bool--false---unionstr-dictstr-any)
- [recognize_ibm(audio_data)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancerecognize_ibmaudio_data-audiodata-username-str-password-str-language-str--en-us-show_all-bool--false---unionstr-dictstr-any)
- [recognize_vosk(audio_data)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancerecognize_voskaudio_data-audiodata--verbose-bool--false---unionstr-dictstr-str)
- [recognize_whisper(audio_data)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancerecognize_whisperaudio_data-audiodata-model-strbase-show_dict-boolfalse-load_optionsnone-transcribe_options)
- [recognize_faster_whisper(audio_data)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancerecognize_faster_whisperaudio_data-audiodata-model-strbase-show_dict-boolfalse-transcribe_options)
- [recognize_openai(audio_data)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancerecognize_openaiaudio_data-audiodata-model--whisper-1-kwargs)
- [recognize_groq(audio_data)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancerecognize_groqaudio_data-audiodata-model--whisper-large-v3-turbo-kwargs)
- [recognize_cohere_api(audio_data)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancerecognize_cohere_apiaudio_data-audiodata--language-str-model--cohere-transcribe-03-2026)

Cada uno de estos métodos admite parámetros opcionales que dependen del motor o API en cuestión. Además cada uno tiene [requisitos específicos](https://pypi.org/project/SpeechRecognition/).

### 4.1 Transcripción de Audio con Vosk (Modo Offline)

Para poder usar Vosk es necesario descargar antes el modelo de reconocimiento que queremos usar de entre la [lista de modelos](https://alphacephei.com/vosk/models) soportados.

```bash
sprc download vosk --url https://alphacephei.com/vosk/models/vosk-model-es-0.42.zip
```

Es importante contemplar una correcta gestión de errores mediante bloques `try/except`. Esto permite cubrir situaciones en las que Vosk no logra interpretar el audio o se presenta algún problema durante la solicitud. 

In [20]:
# Intentamos transcribir el audio usando Vosk (offline)
try:
    transcription = r.recognize_vosk(audio)
    print("Vosk piensa que dijiste: " + transcription)
except sr.UnknownValueError:
    print("Vosk no pudo entender el audio")
except sr.RequestError as e:
    print("Error al solicitar resultados de Vosk; {0}".format(e))

Sphinx piensa que dijiste: computación ubicua e inteligencia ambiental


### 4.2. Reconocimiento de Voz con Google Speech Recognition (Modo Online)

Este método, a diferencia de las alternativas offline, requiere conexión a internet, ya que envía el audio capturado a los servidores de Google para su procesamiento. La ventaja principal es su alta precisión, gracias a los potentes algoritmos y recursos computacionales de Google, lo que suele traducirse en un reconocimiento más exacto y robusto del habla.

Sin embargo, es importante considerar algunas limitaciones de este enfoque:

- **Dependencia de la Conexión a Internet:** Al requerir conexión online, la funcionalidad se ve afectada en entornos sin acceso a la red, lo que puede limitar su uso en aplicaciones móviles o zonas con conectividad inestable.
- **Latencia:** El envío y procesamiento del audio en la nube puede introducir una pequeña demora en el reconocimiento, lo cual podría ser relevante en aplicaciones en tiempo real.
- **Límites y Costos:** Aunque se puede usar gratuitamente para pruebas y aplicaciones de bajo volumen, para un uso intensivo o comercial podría requerirse una suscripción o el uso de una clave de API específica.

En resumen, Google Speech Recognition es una herramienta poderosa para la transcripción de audio con alta precisión, pero su dependencia de una conexión a internet y otros factores deben sopesarse según el contexto de la aplicación.


In [21]:
# Intentamos reconocer el habla utilizando el servicio de Google Speech Recognition
try:
    # Se llama a recognize_google con el audio capturado y se especifica 'es-ES' para el idioma español
    texto = r.recognize_google(audio, language='es-ES')
    # Si se reconoce el audio, se imprime el texto transcrito
    print("Google Speech Recognition cree que dijiste:", texto)
except sr.UnknownValueError:
    # Esta excepción se captura cuando el servicio no logra interpretar el audio
    print("Google Speech Recognition no pudo entender el audio")
except sr.RequestError as e:
    # Esta excepción se maneja en caso de errores en la solicitud (ej. problemas de conectividad)
    print("No se pudieron solicitar resultados del servicio de Google Speech Recognition; {0}".format(e))

Google Speech Recognition cree que dijiste: computación ubicua e inteligencia ambiental


## 5. Reconocimiento en segundo plano

Si queremos utilizar el reconocimiento de voz como método de control de una aplicación (un mecanismo de HCI natural), no podemos bloquear la aplicación mientras de escucha el micrófono o se espera a recibir la transcripción del audio. La solución a este problema es el empleo de concurrencia.

SpeechRecognition ofrece la posibilidad de escuchar en segundo plano mediante el método [listen_in_background(source, callable)](https://github.com/Uberi/speech_recognition/blob/master/reference/library-reference.rst#recognizer_instancelisten_in_backgroundsource-audiosource-callback-callablerecognizer-audiodata-any---callablebool-none)

Este método recibe un argumento *callable* que es una función que será llamada cada vez que se capture audio. Esta función debe recibir dos argumentos: la instancia del recognizer y el AudioData capturado. Su misión es realizar las tareas que necesitemos hacer con ese audio (por ejemplo, transcribir el audio y realizar algunas tareas en función del audio recibido).

La función *listen_in_background* devuelve un objeto que al ser llamado detiene la ejecución en segundo plano.

In [22]:
import time

# Bandera global para detener la escucha cuando se detecta el comando "salir"
exit_flag = False

# Función callback que se ejecuta cada vez que se recibe audio en segundo plano
def procesar_audio(recognizer, audio):
    global exit_flag
    try:
        # Transcribir el audio usando el servicio online de Google Speech Recognition en español
        text = recognizer.recognize_google(audio, language="es-ES")
        print(text)
        
        # Si se detecta el comando "salir", se activa la bandera para finalizar la escucha
        if "salir" in text.lower():
            exit_flag = True

    except sr.UnknownValueError:
        # Manejo de error cuando el servicio no entiende el audio
        print("Google Speech Recognition no pudo entender el audio")
    except sr.RequestError as e:
        # Manejo de error cuando hay problemas en la solicitud, como la falta de conectividad
        print("No se pudieron solicitar resultados del servicio de Google Speech Recognition; {0}".format(e))

# Crear el objeto Recognizer y configurar el Microphone
r = sr.Recognizer()
m = sr.Microphone()

# Calibrar el recognizer al ruido ambiental para una mejor detección
with m as source:
    r.adjust_for_ambient_noise(source)
    print("Calibración completada. Diga 'salir' para terminar...")

# Iniciar la escucha en segundo plano de manera continua
stop_listening = r.listen_in_background(m, procesar_audio)

# Bucle principal que mantiene el script en ejecución hasta que se detecte "salir"
while not exit_flag:
    # Este el es bucle principal donde nuestra aplicación hace el resto de tareas.
    time.sleep(0.1) # Pausa breve para evitar uso excesivo de CPU

# Detener la escucha en segundo plano y finalizar el script
stop_listening()
print("---- Finalizando la escucha en segundo plano. ----")

Calibración completada. Diga 'salir' para terminar...
en un lugar de la Mancha
de cuyo nombre no quiero acordarme
Noa mucho que vivía
un hidalgo caballero
salir
Finalizando la escucha en segundo plano.


## 6. Pasos extra 

### 6.1. Detección del idioma
Una función adicional que se puede implementar en proyectos de reconocimiento de voz es la detección automática de idioma. Esto resulta particularmente útil cuando se esperan entradas de voz en múltiples lenguajes o cuando la aplicación debe adaptarse dinámicamente al idioma que habla el usuario. Algunas bibliotecas de reconocimiento de voz y servicios de procesamiento en la nube ofrecen la posibilidad de identificar el idioma antes de transcribirlo. Esta capacidad facilita la creación de soluciones más versátiles, por ejemplo, aplicaciones de traducción en tiempo real, asistencia virtual multilingüe y sistemas de transcripción internacional.

### 6.2. Consideraciones de memoria y uso de hilos
Aunque el reconocimiento de voz resulta muy potente y abre múltiples posibilidades, también implica un consumo significativo de recursos del sistema, especialmente de memoria y poder de cómputo. A medida que aumentan las demandas de procesamiento (por ejemplo, si reconoces varios idiomas, aplicas modelos de gran tamaño o procesas audio en tiempo real), es probable que debas optimizar el uso de memoria y la velocidad de ejecución. En estos casos, resulta aconsejable configurar hilos o procesos adicionales que trabajen en paralelo para distribuir la carga de trabajo y aprovechar al máximo el hardware disponible. El uso eficiente de hebras (threads) o procesos puede marcar la diferencia en aplicaciones de realidad aumentada o entornos donde se requiera una respuesta rápida y en tiempo real.